In [1]:
from time import process_time

import pandas as pd
import numpy as np

## **Первая модель. Без изменения изначальных признаков.**

_Считаем данные. И оставим только численные колонки._

In [2]:
data = pd.read_csv('5 initial_data.csv', index_col='id')
#выбор нужных колонок
info_cols = ['vendor_id', 'passenger_count', 'pickup_longitude',
                'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'trip_duration']
data = data[info_cols]
data

,vendor_id,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,trip_duration
id,,,,,,,
id2875421,2,1,-73.982155,40.767937,-73.964630,40.765602,455.0
id2377394,1,1,-73.980415,40.738564,-73.999481,40.731152,663.0
id3858529,2,1,-73.979027,40.763939,-74.005333,40.710087,2124.0
id3504673,2,1,-74.010040,40.719971,-74.012268,40.706718,429.0
id2181028,2,1,-73.973053,40.793209,-73.972923,40.782520,435.0
...,...,...,...,...,...,...,...
id2376096,2,4,-73.982201,40.745522,-73.994911,40.740170,778.0
id1049543,1,1,-74.000946,40.747379,-73.970184,40.796547,655.0
id2304944,2,1,-73.959129,40.768799,-74.004433,40.707371,764.0


_Будем считать ошибку модели по **MSLE**._
$$
\text{MSLE}(X, y, a) = \frac{1}{\ell}\sum_{i=1}^{\ell} \big(\log{(y_i + 1)} - \log{(a(x_i) + 1)}\big)^2 \rightarrow min
$$
* _сильнее штрафует за недопредсказание._
* _считает относительную ошибку, а не абсолютную._

_Чтобы нам было легче её считать, мы просто логарифмируем таргетную переменную. А после вычисления предикшена, если будет нужно, применим обратное преобразование логарифма (экспоненту). То есть вернем её в исходный масштаб._


In [3]:
#сначала вычислим логарифм таргета
log_target = np.log1p(data['trip_duration'])
#переназначим столбец
data['trip_duration'] = log_target

_Определимся с признаками и таргетами._

In [4]:
X = data.drop('trip_duration', axis=1)
Y = data['trip_duration']
X

,vendor_id,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
id,,,,,,
id2875421,2,1,-73.982155,40.767937,-73.964630,40.765602
id2377394,1,1,-73.980415,40.738564,-73.999481,40.731152
id3858529,2,1,-73.979027,40.763939,-74.005333,40.710087
id3504673,2,1,-74.010040,40.719971,-74.012268,40.706718
id2181028,2,1,-73.973053,40.793209,-73.972923,40.782520
...,...,...,...,...,...,...
id2376096,2,4,-73.982201,40.745522,-73.994911,40.740170
id1049543,1,1,-74.000946,40.747379,-73.970184,40.796547
id2304944,2,1,-73.959129,40.768799,-74.004433,40.707371


_Выделим конечную тестовую выборку._

In [5]:
from sklearn.model_selection import train_test_split
#тест 20% и сохраним зерно случайности
X_tr, X_ts, Y_tr, Y_ts = train_test_split(X, Y,
                                          test_size=0.2,
                                          random_state=42)

### _Проведем кросс валидацию, используя тренировочную выборку._

In [6]:
from sklearn.model_selection import KFold
#создадим правило для разделения данных
#количество фолдов [трейн, тест] - n_splits
splitter = KFold(n_splits=20, shuffle=True, random_state=33)

_Создадим переменную для хранения замеров качества на каждом из кусочков (фолдов)._

In [7]:
losses = []

_Будем делить данные. Обучать модель. Замерять качество на каждом фолде._

In [8]:
from sklearn.linear_model import LinearRegression
#получим список индексов для разделения
indexes = splitter.split(X_tr)
#чтобы обращаться к X_tr, Y_tr по индексам они должны быть np.array
X_tr_np = X_tr.to_numpy()
Y_tr_np = Y_tr.to_numpy()
#перебираем фолды
for tr_i, val_i in indexes:
    #разделение -=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
    #тренировочная часть текущего фолда
    x_t = X_tr_np[tr_i]
    y_t = Y_tr_np[tr_i]
    #валидация текущего фолда
    x_v = X_tr_np[val_i]
    y_v = Y_tr_np[val_i]

    #обучение -=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
    model = LinearRegression()
    model.fit(x_t, y_t)

    #качество -=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
    pred = model.predict(x_v)
    errors = pred - y_v #вектор
    msle = (errors**2).mean()
    losses.append(msle)

_Посчитаем среднюю ошибку на кросс валидации._

In [9]:
mean_cross_valid_msle = np.mean(losses)
mean_cross_valid_msle

np.float64(0.6134219105328517)

### _Обучим модель на всех тренировочных данных сразу и замерим качество на основной тестовой выборке._

In [10]:
model = LinearRegression()
#обучим модель
model.fit(X_tr, Y_tr)
#замерим её качество
pred = model.predict(X_ts)
error = pred - Y_ts
msle = (error ** 2).mean()
#ошибка на основной тестовой выборке
msle

np.float64(0.6056668879526187)

## **Вторая модель. С изменением изначальных признаков.**

_Считаем данные._

In [11]:
processed_data = pd.read_csv('5 processed_data.csv', index_col='id')
processed_data.head()

,vendor_id,passenger_count,store_and_fwd_flag,trip_duration,distance_km
id,,,,,
id2875421,1,930.399753,0,455.0,1.500479
id2377394,0,930.399753,0,663.0,1.807119
id3858529,1,930.399753,0,2124.0,6.392080
id3504673,1,930.399753,0,429.0,1.487155
id2181028,1,930.399753,0,435.0,1.189925


_Логарифмируем таргет._

In [13]:
#вычислим логарифм
log_dur = np.log1p(processed_data['trip_duration'])
#создадим новый столбец
processed_data = processed_data.assign(
    log_trip_duration=log_dur)
#удалим старый столбец
processed_data.drop(['trip_duration'], axis=1)
processed_data['log_trip_duration']

id
id2875421    6.122493
id2377394    6.498282
id3858529    7.661527
id3504673    6.063785
id2181028    6.077642
               ...   
id2376096    6.658011
id1049543    6.486161
id2304944    6.639876
id2714485    5.924256
id1209952    5.293305
Name: log_trip_duration, Length: 1458644, dtype: float64

_Разделим данные на признаки и таргет._

In [14]:
X_2 = processed_data.drop(['log_trip_duration'], axis=1)
Y_2 = processed_data['log_trip_duration']

_Разделим данные на трейн и тест. Используя индексы (в нашем случае id, полученные после прошлого разбиения)._

In [15]:
ts_id = X_ts.index
tr_id = X_tr.index

_Возьмем строки с тем же индексом(у нас это id)._
* _iloc - поиск по номерным индексам (позиция)_
* _loc - поиск по названным индексам_

In [16]:
#тренировочная
X_tr_2 = X_2.loc[tr_id]
Y_tr_2 = Y_2.loc[tr_id]

#тестовая
X_ts_2 = X_2.loc[ts_id]
Y_ts_2 = Y_2.loc[ts_id]

_Проведем кросс-валидацию с помощью метода cross_validate()._
* _scoring - с помощью чего оценивать_
* _cv - способ разделения_
* _return_train_score - хотим возвращать не только качество проверки, но и качество обучения._

In [17]:
from sklearn.model_selection import cross_validate
#создадим модель
model = LinearRegression()
#проведем её кросс валидацию
cv_result = cross_validate(model, X_tr_2, Y_tr_2,
                           scoring='neg_mean_squared_error',
                           cv=splitter,
                           return_train_score=True)
#cv_result содержит информацию о каждом фолде: время, а также результаты теста и трейна

_Замерим среднее качество тестов на кросс-валидации. Также не забыв снова сделать их положительными, так как cross_validate возвращает их с минусом._

In [18]:
np.mean(-cv_result['test_score'])

np.float64(0.41744701080889657)

_Теперь обучим модель на всех тренировочных данных._

In [19]:
model = LinearRegression()
model.fit(X_tr_2, Y_tr_2)
#вычислим её предсказание
mod2_pred = model.predict(X_ts_2)
#вычислим её среднюю ошибку
np.mean((mod2_pred - Y_ts_2)**2)

np.float64(0.372656083176397)